In [1]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
import json
import os
import sys

from pyspark.sql import Row

from src.api.api_client import APIClient
from src.utils.generate_params import GenerateParams
from src.ingestion.table_manager import IcebergTableManager
from settings import TOKEN_API, URL_API

In [2]:
spark

In [3]:
manager = IcebergTableManager(spark,'quote_list','bronze','grand_line')

DataFrame[]

In [ ]:
manager.create_table()

In [4]:
range_search = '5d'
interval = '1d'
fundamental = 'false'
dividend = 'false'
modules = 'summaryProfile'
acoes = ['LMTB34', 'NOCG34']
for acoe in acoes:
    print(acoe)
    url_api = f"https://brapi.dev/api/quote/{acoe}"
    generate_parans = GenerateParams(acoe, range_search, interval, fundamental, dividend, modules, TOKEN_API)
    params = generate_parans.get_params()
    print(params)
    api_request = APIClient(url_api, params)
    result = api_request.request()['results'][0]
    for key in ['validRanges', 'validIntervals', 'usedInterval', 'usedRange']:
        result.pop(key, None)  # Evita erro caso a chave já tenha sido removida
    
    # Converter o dicionário para JSON
    json_data = json.dumps(result)

    # Criar DataFrame com a coluna JSON
    df = spark.createDataFrame([Row(raw=json_data)])

    # Gravar no banco
    df.write.mode("append").saveAsTable("raw.db.raw_data")


LMTB34
{'range': '5d', 'interval': '1d', 'fundamental': 'false', 'dividends': 'false', 'modules': 'summaryProfile', 'token': '6AgYDFoKXU67YGpV9TsGZP'}


NOCG34
{'range': '5d', 'interval': '1d', 'fundamental': 'false', 'dividends': 'false', 'modules': 'summaryProfile', 'token': '6AgYDFoKXU67YGpV9TsGZP'}


In [5]:
df = spark.sql("SELECT * FROM demo.raw.db.raw_data")
df.show(truncate=True)  # Mostra os dados sem cortar o JSON


+--------------------+
|                 raw|
+--------------------+
|{"currency": "BRL...|
|{"currency": "BRL...|
+--------------------+

